# Device Failure Probability Prediction

This notebook does two things simultaneously from a single model:
- **Regression accuracy** → predicts a continuous [0,1] failure probability (MAE, RMSE, R²)
- **Recall evaluation** → converts predictions to binary at a threshold to check if real failures are being caught

It also checks whether the training data is biased (unevenly distributed) and corrects it before training.

In [2]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.model_selection import train_test_split

df = pd.read_csv('device_failure_dataset.csv')
df.head()

,device_id,timestamp,cpu_utilization,memory_pressure,temperature_anomaly_score,packet_loss_rate,firmware_version,software_patch_age_days,device_age_days,maintenance_days_since,...,historical_incidents,historical_outages,sla_breach_history,error_log_rate,reboot_count_7d,network_latency,io_wait,power_cycle_count,temperature_variance,failure_probability
0,D8359,2025-08-26,0.68,0.12,0.28,0.033,3.5,346,2333,44,...,1,0,1,10.9,0,297,0.20,20,0.35,0.38
1,D3045,2025-01-22,0.35,0.88,0.76,0.024,5.2,174,1238,79,...,10,1,1,19.0,11,186,0.60,25,0.02,0.49
2,D6820,2024-08-02,0.17,0.36,0.63,0.133,5.1,295,887,360,...,7,4,1,42.8,3,204,0.28,20,0.42,0.43
3,D2084,2024-08-04,0.70,0.73,0.68,0.011,4.0,273,1102,83,...,8,10,8,11.0,10,38,0.23,1,0.40,0.53
4,D1771,2024-04-22,0.92,0.61,0.72,0.032,6.0,202,2733,234,...,4,3,8,26.9,13,214,0.36,4,0.25,0.57


## 1. Prepare Features

In [3]:
# Remove identifier columns that carry no predictive signal.
# device_id is just a label; timestamp is not used for time-series here.
drop_cols = ['device_id', 'timestamp']
df_model = df.drop(columns=drop_cols)

model_target   = 'failure_probability'
model_features = df_model.columns.drop(model_target).tolist()

print('Model features:', model_features)
print('Model target  :', model_target)
print('Data shape    :', df_model.shape)
df_model.describe()

Model features: ['cpu_utilization', 'memory_pressure', 'temperature_anomaly_score', 'packet_loss_rate', 'firmware_version', 'software_patch_age_days', 'device_age_days', 'maintenance_days_since', 'workload_intensity', 'historical_incidents', 'historical_outages', 'sla_breach_history', 'error_log_rate', 'reboot_count_7d', 'network_latency', 'io_wait', 'power_cycle_count', 'temperature_variance']
Model target  : failure_probability
Data shape    : (2500, 19)


,cpu_utilization,memory_pressure,temperature_anomaly_score,packet_loss_rate,firmware_version,software_patch_age_days,device_age_days,maintenance_days_since,workload_intensity,historical_incidents,historical_outages,sla_breach_history,error_log_rate,reboot_count_7d,network_latency,io_wait,power_cycle_count,temperature_variance,failure_probability
count,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000
mean,0.552156,0.546772,0.502996,0.075446,4.556080,181.309600,1565.421200,180.136400,0.551080,10.161200,5.096000,5.076000,25.045080,7.485600,157.338000,0.499868,14.976000,0.257348,0.523180
std,0.259625,0.253460,0.288929,0.043052,0.922837,104.840526,824.333599,105.051919,0.264279,5.910498,3.151056,3.171474,14.541255,4.699726,83.646739,0.291864,9.070983,0.144722,0.099169
min,0.100000,0.100000,0.000000,0.000000,3.000000,0.000000,102.000000,0.000000,0.100000,0.000000,0.000000,0.000000,0.100000,0.000000,10.000000,0.000000,0.000000,0.000000,0.210000
25%,0.330000,0.330000,0.260000,0.039000,4.000000,89.000000,876.750000,88.750000,0.320000,5.000000,2.000000,2.000000,12.600000,3.000000,84.000000,0.240000,7.000000,0.130000,0.460000
50%,0.550000,0.540000,0.510000,0.077000,5.000000,182.500000,1565.500000,178.000000,0.540000,10.000000,5.000000,5.000000,24.900000,8.000000,157.000000,0.495000,15.000000,0.260000,0.520000
75%,0.780000,0.760000,0.750000,0.112000,5.200000,270.000000,2265.000000,272.000000,0.780000,15.000000,8.000000,8.000000,37.825000,12.000000,231.000000,0.750000,23.000000,0.380000,0.590000
max,1.000000,1.000000,1.000000,0.150000,6.000000,365.000000,2994.000000,364.000000,1.000000,20.000000,10.000000,10.000000,50.000000,15.000000,300.000000,1.000000,30.000000,0.500000,0.850000


## 2. Data Bias Check & Fix

**What is data bias here?**
If most devices in the dataset have a very low failure probability (e.g. 90% of rows have failure_probability < 0.3),
the model can cheat — it learns to always predict 'low risk' and still looks accurate because most data is low risk.
This is the 'accuracy illusion': the model appears to perform well but completely misses real failures.

**How we detect it:** We check the skewness of the target and the class balance after applying a failure threshold.
- Skewness > 1 or < -1 means the distribution is significantly lopsided.
- Class imbalance > 80/20 split means the model will be biased toward the majority class.

**How we fix it:** We oversample the minority class (high-risk devices) in the training set using SMOTE
(Synthetic Minority Oversampling Technique), which generates synthetic high-risk examples so the model
gets an equal chance to learn from both low-risk and high-risk devices.
Note: we ONLY apply this fix to the training data — the test set stays untouched so evaluation is realistic.

In [6]:
# --- Step 1: Check the distribution of the target variable ---
print('=== Target Distribution Check ===')
print(f'Mean     : {df_model[model_target].mean():.4f}')
print(f'Median   : {df_model[model_target].median():.4f}')
print(f'Std Dev  : {df_model[model_target].std():.4f}')
print(f'Skewness : {df_model[model_target].skew():.4f}')
print()

# Interpret skewness
skew = df_model[model_target].skew()
if abs(skew) < 0.5:
    print('Skewness is low — target distribution is fairly balanced.')
elif abs(skew) < 1.0:
    print('Moderate skew detected — distribution is somewhat lopsided.')
else:
    print('High skew detected — distribution is heavily imbalanced. Fixing required.')

# --- Step 2: Check class balance after applying a 0.5 threshold ---
# This mimics what happens when we convert predictions to binary for recall evaluation.
THRESHOLD = 0.5
binary_labels = (df_model[model_target] >= THRESHOLD).astype(int)
class_counts  = binary_labels.value_counts()
total         = len(binary_labels)

print(f'\n=== Class Balance (at threshold = {THRESHOLD}) ===')
print(f'No Failure (0): {class_counts.get(0, 0)}  ({class_counts.get(0, 0)/total:.1%})')
print(f'Will Fail  (1): {class_counts.get(1, 0)}  ({class_counts.get(1, 0)/total:.1%})')

majority_pct = class_counts.max() / total
if majority_pct > 0.80:
    print('Severe imbalance detected (>80/20). SMOTE oversampling will be applied to training data.')
elif majority_pct > 0.65:
    print('Moderate imbalance detected (>65/35). SMOTE oversampling will be applied to training data.')
else:
    print('Class balance is acceptable — no oversampling needed.')

=== Target Distribution Check ===
Mean     : 0.5232
Median   : 0.5200
Std Dev  : 0.0992
Skewness : 0.0189

Skewness is low — target distribution is fairly balanced.

=== Class Balance (at threshold = 0.5) ===
No Failure (0): 983  (39.3%)
Will Fail  (1): 1517  (60.7%)
Class balance is acceptable — no oversampling needed.


In [7]:
# --- Step 3: Split first, then fix ONLY the training set ---
# IMPORTANT: We split before applying any fix so that the test set
# reflects the real-world distribution. Fixing the test set would
# give an unrealistically optimistic evaluation.

train_raw, test = train_test_split(df_model, test_size=0.25, shuffle=True)

print(f'Train (before fix) : {len(train_raw)} rows')
print(f'Test  (untouched)  : {len(test)} rows')

Train (before fix) : 1875 rows
Test  (untouched)  : 625 rows


In [8]:
!pip install imbalanced-learn -q

In [10]:
from imblearn.over_sampling import SMOTE

# SMOTE needs a binary label to know which class to oversample.
# We create a temporary binary column from the continuous target.
X_train_raw = train_raw[model_features]
y_train_raw = train_raw[model_target]
y_train_bin = (y_train_raw >= THRESHOLD).astype(int)   # temporary binary version

# Check if fixing is needed (imbalance > 65/35)
majority_train_pct = y_train_bin.value_counts().max() / len(y_train_bin)

if majority_train_pct > 0.65:
    print(f'Applying SMOTE — majority class is {majority_train_pct:.1%} of training data...')

    # SMOTE generates new synthetic minority-class rows by interpolating
    # between existing minority examples, rather than just duplicating them.
    # This gives the model more variety to learn from.
    smote = SMOTE(random_state=None)   # random_state=None keeps it randomized each run
    X_resampled, y_bin_resampled = smote.fit_resample(X_train_raw, y_train_bin)

    # SMOTE only works on binary labels, so we reconstruct the continuous
    # failure_probability for the resampled rows:
    # - Original rows keep their real probability value
    # - Synthetic rows get a probability assigned based on their binary class
    #   (class 0 → random low value, class 1 → random high value)
    n_original  = len(X_train_raw)
    n_synthetic = len(X_resampled) - n_original

    original_probs  = y_train_raw.values
    synthetic_probs = np.where(
        y_bin_resampled[n_original:] == 1,
        np.random.uniform(THRESHOLD, 1.0, n_synthetic),    # high-risk synthetic
        np.random.uniform(0.0, THRESHOLD, n_synthetic)     # low-risk synthetic
    )
    y_resampled = np.concatenate([original_probs, synthetic_probs])

    # Reassemble into a clean training DataFrame
    train = pd.DataFrame(X_resampled, columns=model_features)
    train[model_target] = y_resampled

    print(f'Training set after SMOTE : {len(train)} rows')
    new_bin = (train[model_target] >= THRESHOLD).astype(int)
    print(f'New class balance — No Failure: {(new_bin==0).sum()}  Will Fail: {(new_bin==1).sum()}')

else:
    print('No fix needed — using original training data.')
    train = train_raw

y_train = train[model_target]
y_test  = test[model_target]

No fix needed — using original training data.


## 3. Helper Functions — Regression & Recall Metrics

We define two separate evaluation functions:
- `regression_metrics`: measures how close the predicted probability is to the actual value
- `recall_metrics`: measures how well we detect real failures after converting to binary with a threshold

In [12]:
def regression_metrics(y_true, y_pred):
    """
    Regression accuracy — evaluates the model's predicted probabilities directly.

    MAE  (Mean Absolute Error): average gap between predicted and actual probability.
           Lower is better. An MAE of 0.05 means we're off by ~5 percentage points on average.
    RMSE (Root Mean Squared Error): like MAE but penalises large errors more heavily.
           Lower is better.
    R²   (R-squared): how much of the variation in failure probability the model explains.
           1.0 = perfect, 0.0 = no better than guessing the mean, negative = worse than guessing.
    """
    mae  = metrics.mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(metrics.mean_squared_error(y_true, y_pred))
    r2   = metrics.r2_score(y_true, y_pred)
    print(f'  MAE  : {mae:.4f}   (avg error in probability units — lower is better)')
    print(f'  RMSE : {rmse:.4f}   (penalises large errors more — lower is better)')
    print(f'  R\u00b2   : {r2:.4f}   ({r2:.1%} of variance explained — higher is better)')


def recall_metrics(y_true_prob, y_pred_prob, threshold=0.5):
    """
    Recall evaluation — converts predicted probabilities to binary at a threshold,
    then measures how well we catch real failures.

    Recall    : of all devices that actually failed, what fraction did we flag? (most important)
    Precision : of all devices we flagged, what fraction actually failed?
    F1        : harmonic mean of recall and precision — balances both.

    Confusion matrix:
      TN = correctly predicted safe   FP = falsely flagged as failing (false alarm)
      FN = missed real failures   TP = correctly caught failures

    For failure prediction, minimising FN (missed failures) is the priority.
    Lowering the threshold increases recall but also increases FP (false alarms).
    """
    y_true_bin = (y_true_prob >= threshold).astype(int)
    y_pred_bin = (y_pred_prob >= threshold).astype(int)

    recall    = metrics.recall_score(y_true_bin, y_pred_bin, zero_division=0)
    precision = metrics.precision_score(y_true_bin, y_pred_bin, zero_division=0)
    f1        = metrics.f1_score(y_true_bin, y_pred_bin, zero_division=0)
    cm        = metrics.confusion_matrix(y_true_bin, y_pred_bin)

    print(f'  Threshold : {threshold}')
    print(f'  Recall    : {recall:.4f}   (of all real failures, how many did we catch?)')
    print(f'  Precision : {precision:.4f}   (of flagged devices, how many actually fail?)')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  Confusion Matrix:')
    print(f'    TN={cm[0,0]}  FP={cm[0,1]}')
    print(f'    FN={cm[1,0]}  TP={cm[1,1]}')
    print(f'  -> Missed failures (FN): {cm[1,0]}  |  False alarms (FP): {cm[0,1]}')

print('Helper functions ready.')

Helper functions ready.


## 4. AutoGluon — Automated Regression

AutoGluon automatically trains many model types (LightGBM, XGBoost, CatBoost, Neural Nets, Random Forests, etc.),
tunes their hyperparameters, and builds an ensemble of the best ones. You don't need to pick a model manually.

We use `problem_type='regression'` so it outputs continuous probabilities in [0,1] rather than classes.
The eval metric `r2` (R-squared) guides AutoGluon to optimise probability accuracy during training.

In [13]:
!pip install autogluon.tabular -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.2 MB/s eta 0:00:00


In [14]:
from autogluon.tabular import TabularDataset, TabularPredictor

train_ag = TabularDataset(train)
test_ag  = TabularDataset(test)

predictor = TabularPredictor(
    label=model_target,
    eval_metric='r2',             # R² guides AutoGluon to optimise probability accuracy
    problem_type='regression',    # continuous [0,1] output, not binary classes
    path='ag_device_failure_reg/'
).fit(
    train_ag,
    presets='best_quality',       # tries all model types and ensembles the best ones
                                  # swap to 'medium_quality' for faster experimentation
    time_limit=300                # seconds — increase for better results on larger datasets
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.10.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.17 GB / 12.67 GB (88.2%)
Disk Space Avail:   86.32 GB / 107.72 GB (80.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` v

## 5. Regression Accuracy — Training vs Testing

We evaluate the model's raw probability predictions on both the training set and the unseen test set.
Comparing the two tells us whether the model has overfit:
- If training R² >> testing R², the model memorised training data and won't generalise.
- A small gap (< 5%) means the model has learned real patterns and should perform well on new devices.

In [15]:
train_preds = predictor.predict(train_ag).clip(0, 1)
test_preds  = predictor.predict(test_ag).clip(0, 1)

# .clip(0, 1) ensures predictions stay within valid probability range
# in case any model slightly overshoots due to ensemble averaging

print('=' * 55)
print('   REGRESSION ACCURACY — How Close Are Probabilities?')
print('=' * 55)

print('\n--- On Training Data (data the model has seen) ---')
regression_metrics(y_train, train_preds)

print('\n--- On Testing Data (data the model has never seen) ---')
regression_metrics(y_test, test_preds)

train_r2 = metrics.r2_score(y_train, train_preds)
test_r2  = metrics.r2_score(y_test, test_preds)
gap = train_r2 - test_r2
print(f'\n--- Overfitting Check ---')
print(f'  Train R\u00b2 - Test R\u00b2 = {gap:.4f}  (\u2248 {gap:.1%})')
if gap < 0.05:
    print('  Small gap — model generalises well to unseen data.')
elif gap < 0.15:
    print('  Moderate gap — some overfitting, consider reducing time_limit or preset.')
else:
    print('  Large gap — model is overfitting. Try medium_quality preset or more data.')

   REGRESSION ACCURACY — How Close Are Probabilities?

--- On Training Data (data the model has seen) ---
  MAE  : 0.0175   (avg error in probability units — lower is better)
  RMSE : 0.0208   (penalises large errors more — lower is better)
  R²   : 0.9555   (95.5% of variance explained — higher is better)

--- On Testing Data (data the model has never seen) ---
  MAE  : 0.0257   (avg error in probability units — lower is better)
  RMSE : 0.0301   (penalises large errors more — lower is better)
  R²   : 0.9116   (91.2% of variance explained — higher is better)

--- Overfitting Check ---
  Train R² - Test R² = 0.0439  (≈ 4.4%)
  Small gap — model generalises well to unseen data.


## 6. Recall Evaluation: Are We Catching Real Failures?

Regression accuracy tells us how close our probabilities are, but for failure prediction we also need to know:
**'Of all the devices that actually failed, how many did our model flag?'** — that is recall.

To compute recall, we apply a threshold to convert continuous probabilities into binary decisions:
- Predicted probability >= threshold → 'WILL FAIL'
- Predicted probability <  threshold → 'SAFE'

We evaluate recall on the test set (unseen data) because that is what matters in production.
We also show training recall so you can spot if the model is over-confident on data it has seen.

In [16]:
# Default threshold — adjust based on your risk tolerance.
# Lower threshold = catch more failures (higher recall) but more false alarms.
# Higher threshold = fewer false alarms but more missed failures.
THRESHOLD = 0.5

print('=' * 55)
print('   RECALL EVALUATION — Are We Catching Real Failures?')
print('=' * 55)

print('\n--- On Training Data ---')
recall_metrics(y_train, train_preds, threshold=THRESHOLD)

print('\n--- On Testing Data (most important) ---')
recall_metrics(y_test, test_preds, threshold=THRESHOLD)

   RECALL EVALUATION — Are We Catching Real Failures?

--- On Training Data ---
  Threshold : 0.5
  Recall    : 0.9276   (of all real failures, how many did we catch?)
  Precision : 0.9546   (of flagged devices, how many actually fail?)
  F1 Score  : 0.9409
  Confusion Matrix:
    TN=692  FP=50
    FN=82  TP=1051
  -> Missed failures (FN): 82  |  False alarms (FP): 50

--- On Testing Data (most important) ---
  Threshold : 0.5
  Recall    : 0.9089   (of all real failures, how many did we catch?)
  Precision : 0.9184   (of flagged devices, how many actually fail?)
  F1 Score  : 0.9136
  Confusion Matrix:
    TN=210  FP=31
    FN=35  TP=349
  -> Missed failures (FN): 35  |  False alarms (FP): 31


**7. Threshold Tuning**

The threshold determines when the model flags a device as at-risk. Since a missed failure costs far more than a false alarm, the threshold should lean lower — start at 0.4 and raise it only if the volume of flagged devices becomes unmanageable.

**Reading the table:**
- `recall` — share of real failures caught
- `precision` — share of flagged devices that actually fail
- `flagged_%` — proportion of your fleet alerted at that threshold

In [17]:
# We evaluate thresholds against the test set only — that is the real-world performance.
y_true_bin = (y_test >= 0.5).astype(int)

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    t = round(t, 2)
    y_pred_bin = (test_preds >= t).astype(int)
    recall    = metrics.recall_score(y_true_bin, y_pred_bin, zero_division=0)
    precision = metrics.precision_score(y_true_bin, y_pred_bin, zero_division=0)
    f1        = metrics.f1_score(y_true_bin, y_pred_bin, zero_division=0)
    flagged   = y_pred_bin.sum() / len(y_pred_bin)
    rows.append({
        'threshold'  : t,
        'recall'     : round(recall, 4),
        'precision'  : round(precision, 4),
        'f1'         : round(f1, 4),
        'flagged_%'  : f'{flagged:.1%}'
    })

threshold_df = pd.DataFrame(rows)
print('Threshold tuning on TEST data:')
threshold_df

Threshold tuning on TEST data:


,threshold,recall,precision,f1,flagged_%
0,0.20,1.0000,0.6144,0.7611,100.0%
1,0.25,1.0000,0.6144,0.7611,100.0%
2,0.30,1.0000,0.6154,0.7619,99.8%
3,0.35,1.0000,0.6379,0.7789,96.3%
4,0.40,1.0000,0.6761,0.8067,90.9%
5,0.45,1.0000,0.7869,0.8807,78.1%
6,0.50,0.9089,0.9184,0.9136,60.8%
7,0.55,0.6094,1.0000,0.7573,37.4%
8,0.60,0.3568,1.0000,0.5259,21.9%
9,0.65,0.1615,1.0000,0.2780,9.9%


## 8. Model Leaderboard

AutoGluon trains many models internally. The leaderboard ranks all of them by test set R².
The top row is the model AutoGluon used for predictions above.

In [18]:
predictor.leaderboard(test_ag, silent=False)

                     model  score_test  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      WeightedEnsemble_L3    0.911959   0.901544          r2        1.720349       1.110595  155.123327                 0.003932                0.001069           0.169883            3       True         15
1      WeightedEnsemble_L2    0.911585   0.902872          r2        0.729408       0.316325  104.346837                 0.003471                0.001016           0.105630            2       True          8
2   NeuralNetFastAI_BAG_L1    0.910025   0.895745          r2        0.177872       0.107775   18.514716                 0.177872                0.107775          18.514716            1       True          5
3   RandomForestMSE_BAG_L2    0.909619   0.892299          r2        1.533405       1.009259  142.771531                 0.200948                0.204548          10.64

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,0.911959,0.901544,r2,1.720349,1.110595,155.123327,0.003932,0.001069,0.169883,3,True,15
1,WeightedEnsemble_L2,0.911585,0.902872,r2,0.729408,0.316325,104.346837,0.003471,0.001016,0.105630,2,True,8
2,NeuralNetFastAI_BAG_L1,0.910025,0.895745,r2,0.177872,0.107775,18.514716,0.177872,0.107775,18.514716,1,True,5
3,RandomForestMSE_BAG_L2,0.909619,0.892299,r2,1.533405,1.009259,142.771531,0.200948,0.204548,10.646544,2,True,11
4,ExtraTreesMSE_BAG_L2,0.909104,0.895791,r2,1.534310,1.022941,134.748867,0.201852,0.218230,2.623880,2,True,12
5,NeuralNetTorch_BAG_L1,0.909006,0.895425,r2,0.087681,0.056427,41.752587,0.087681,0.056427,41.752587,1,True,7
6,NeuralNetFastAI_BAG_L2,0.908909,0.888056,r2,1.514565,0.891296,152.329565,0.182107,0.086586,20.204578,2,True,13
7,XGBoost_BAG_L2,0.906917,0.888489,r2,1.452835,0.839356,145.095015,0.120377,0.034646,12.970028,2,True,14
8,LightGBM_BAG_L2,0.905897,0.890976,r2,1.392471,0.823909,145.046618,0.060014,0.019198,12.921631,2,True,10
9,LightGBMXT_BAG_L2,0.905841,0.888024,r2,1.412935,0.830505,141.312785,0.080478,0.025794,9.187798,2,True,9


## 9. Feature Importance

Shows which features contributed most to the model's predictions.
Higher importance = the model relies on that feature more to predict failure probability.
This is useful for understanding what drives failures and for removing uninformative features.

In [19]:
importance = predictor.feature_importance(test_ag)
print('Top 10 most important features:')
print(importance.head(10))

Computing feature importance via permutation shuffling for 18 features using 625 rows with 5 shuffle sets...
	72.43s	= Expected runtime (14.49s per shuffle set)
	37.64s	= Actual runtime (Completed 5 of 5 shuffle sets)


Top 10 most important features:
                           importance    stddev       p_value  n  p99_high  \
cpu_utilization              0.518582  0.033949  2.191593e-06  5  0.588485   
temperature_anomaly_score    0.347338  0.017572  7.833825e-07  5  0.383519   
memory_pressure              0.272442  0.006311  3.452886e-08  5  0.285437   
packet_loss_rate             0.161197  0.006213  2.643178e-07  5  0.173990   
workload_intensity           0.136735  0.006605  6.512634e-07  5  0.150334   
historical_incidents         0.044424  0.002906  2.183792e-06  5  0.050407   
error_log_rate               0.044322  0.002249  7.929926e-07  5  0.048953   
software_patch_age_days      0.043722  0.003444  4.580238e-06  5  0.050813   
reboot_count_7d              0.040380  0.004049  1.197152e-05  5  0.048717   
device_age_days              0.036773  0.002991  5.202897e-06  5  0.042930   

                            p99_low  
cpu_utilization            0.448680  
temperature_anomaly_score  0.3111

## 10. Predict on a New Device

Pass in any device's current metrics to get its failure probability score.
Adjust ALERT_THRESHOLD based on the threshold tuning table above.

In [20]:
# Set this to the threshold you chose from Section 7
ALERT_THRESHOLD = 0.5

new_device = pd.DataFrame([{
    'cpu_utilization'           : 0.83,
    'memory_pressure'           : 0.71,
    'temperature_anomaly_score' : 0.64,
    'packet_loss_rate'          : 0.03,
    'firmware_version'          : 5.1,
    'software_patch_age_days'   : 120,
    'device_age_days'           : 1450,
    'maintenance_days_since'    : 95,
    'workload_intensity'        : 0.88,
    'historical_incidents'      : 4,
    'historical_outages'        : 1,
    'sla_breach_history'        : 2,
    'error_log_rate'            : 12,
    'reboot_count_7d'           : 3,
    'network_latency'           : 84,
    'io_wait'                   : 0.41,
    'power_cycle_count'         : 9,
    'temperature_variance'      : 0.22
}])

score  = predictor.predict(new_device).clip(0, 1).values[0]
action = 'ALERT: Schedule maintenance' if score >= ALERT_THRESHOLD else 'OK: No action needed'

print(f'Failure Probability : {score:.4f}  ({score:.1%})')
print(f'Status              : {action}')

Failure Probability : 0.5610  (56.1%)
Status              : ALERT: Schedule maintenance
